# Attention Weight Visualization

Loads the latest checkpoint produced by `training_together.ipynb` and visualises
per-head attention patterns, mirroring the approach in
[HelloWorld_Transformer_with_RoPE](https://github.com/ramayer/google-colab-examples/blob/main/HelloWorld_Transformer_with_RoPE.ipynb).

`model.forward(token_ids, return_attention_weights=True)` returns
`(logits, all_attn_weights)` where `all_attn_weights` is a list of tensors —
one per layer — each of shape `(batch, heads, seq_len, seq_len)`.

In [ ]:
import pathlib
import re
import torch
import tiktoken
import matplotlib.pyplot as plt
import numpy as np
import sys
sys.path.insert(0, '..')

from cs336_basics.modules import TransformerLanguageModel, load_checkpoint, AdamW

## Locate the latest checkpoint

In [ ]:
def find_latest_checkpoint(*search_dirs):
    """Return the checkpoint_step_*_loss.pt file with the highest step number."""
    candidates = []
    for d in search_dirs:
        candidates.extend(pathlib.Path(d).glob("checkpoint_step_*_loss.pt"))
    if not candidates:
        raise FileNotFoundError(
            f"No checkpoint files found in {search_dirs}. "
            "Run training_together.ipynb first."
        )
    def step(p):
        m = re.search(r'checkpoint_step_(\d+)_loss', p.name)
        return int(m.group(1)) if m else -1
    best = max(candidates, key=step)
    print(f"Latest checkpoint: {best}  (step {step(best)})")
    return best

# Checkpoints are written relative to wherever the training notebook ran.
# We search the repo root and the notebooks/ directory.
REPO_ROOT   = pathlib.Path('..').resolve()
NB_DIR      = pathlib.Path('.').resolve()
checkpoint_path = find_latest_checkpoint(REPO_ROOT, NB_DIR)

## Recreate the model with the same hyperparameters as `training_together.ipynb`

In [ ]:
# ── Must match training_together.ipynb exactly ──────────────────────────────
ENCODING_NAME = "r50k_base"
CONTEXT_LENGTH = 256
NUM_LAYERS     = 4
D_MODEL        = 512
NUM_HEADS      = 16
D_FF           = 1344
ROPE_THETA     = 10000
# ─────────────────────────────────────────────────────────────────────────────

tokenizer  = tiktoken.get_encoding(ENCODING_NAME)
VOCAB_SIZE = tokenizer.n_vocab          # 50257

rope_params = {"theta": ROPE_THETA, "max_seq_len": CONTEXT_LENGTH}

model = TransformerLanguageModel(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    d_ff=D_FF,
    num_layers=NUM_LAYERS,
    max_seq_len=CONTEXT_LENGTH,
    rope_params=rope_params,
)

# load_checkpoint requires an optimizer instance even though we don't use it for inference
dummy_optimizer = AdamW(model.parameters(), lr=1e-3)
iteration = load_checkpoint(checkpoint_path, model, dummy_optimizer)
model.eval()
print(f"Loaded model from step {iteration}")

## Encode a prompt and run inference

In [ ]:
PROMPT = "Once upon a time"

token_ids = torch.tensor(
    tokenizer.encode(PROMPT, allowed_special={"<|endoftext|>"}),
    dtype=torch.long,
).unsqueeze(0)  # (1, seq_len)

# Pad / trim to CONTEXT_LENGTH so attention maps are square
if token_ids.shape[1] < CONTEXT_LENGTH:
    pad = torch.zeros(1, CONTEXT_LENGTH - token_ids.shape[1], dtype=torch.long)
    token_ids = torch.cat([token_ids, pad], dim=1)
else:
    token_ids = token_ids[:, :CONTEXT_LENGTH]

tokens_str = [tokenizer.decode([t]) for t in token_ids[0].tolist()]
print(f"Sequence length: {token_ids.shape[1]}")
print("Tokens:", tokens_str[:20], "...")

with torch.no_grad():
    logits, all_attn_weights = model(token_ids, return_attention_weights=True)

print(f"logits shape:           {logits.shape}")
print(f"num layers:             {len(all_attn_weights)}")
print(f"attn_weights[0] shape:  {all_attn_weights[0].shape}  (batch, heads, seq, seq)")

## Visualise — all heads for each layer

Each subplot is one attention head.
Rows = query positions, columns = key positions.
The causal mask means the upper triangle is always zero.
Only the first `N_SHOW` tokens are displayed to keep the plots readable.

In [ ]:
def plot_attention_weights(all_attn_weights, tokens=None, n_show=32, cmap='viridis'):
    """
    Grid of attention heatmaps: rows=layers, columns=heads.

    Parameters
    ----------
    all_attn_weights : list[Tensor]
        One tensor per layer, shape (batch, heads, seq_len, seq_len).
    tokens : list[str] or None
        Optional token-string labels for the axes.
    n_show : int
        Number of token positions to display (truncated for readability).
    """
    num_layers = len(all_attn_weights)
    num_heads  = all_attn_weights[0].shape[1]
    n_show     = min(n_show, all_attn_weights[0].shape[2])
    tick_labels = (tokens[:n_show] if tokens else [str(i) for i in range(n_show)])

    fig, axes = plt.subplots(
        num_layers, num_heads,
        figsize=(3 * num_heads, 3 * num_layers),
        squeeze=False,
    )
    for layer_idx, attn in enumerate(all_attn_weights):
        attn_np = attn[0, :, :n_show, :n_show].detach().cpu().numpy()  # (heads, n, n)
        for head_idx in range(num_heads):
            ax = axes[layer_idx][head_idx]
            im = ax.imshow(attn_np[head_idx], vmin=0, vmax=1, cmap=cmap, aspect='auto')
            ax.set_title(f'L{layer_idx} H{head_idx}', fontsize=8)
            ax.set_xticks(range(n_show))
            ax.set_yticks(range(n_show))
            ax.set_xticklabels(tick_labels, rotation=90, fontsize=6)
            ax.set_yticklabels(tick_labels, fontsize=6)
            if head_idx == 0:
                ax.set_ylabel('Query', fontsize=8)
            if layer_idx == num_layers - 1:
                ax.set_xlabel('Key', fontsize=8)

    fig.suptitle(
        f'Attention weights — prompt: "{PROMPT}"  (step {iteration})',
        fontsize=12, y=1.01,
    )
    plt.colorbar(im, ax=axes, shrink=0.5, label='Attention weight')
    plt.tight_layout()
    plt.show()


plot_attention_weights(all_attn_weights, tokens=tokens_str)

## Single layer / single head deep-dive

In [ ]:
LAYER = 0
HEAD  = 0
N_SHOW = 32

w = all_attn_weights[LAYER][0, HEAD, :N_SHOW, :N_SHOW].detach().cpu().numpy()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(w, vmin=0, vmax=1, cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_title(f'Attention — Layer {LAYER}, Head {HEAD}  |  "{PROMPT}"')
ax.set_xlabel('Key position')
ax.set_ylabel('Query position')
ax.set_xticks(range(N_SHOW))
ax.set_yticks(range(N_SHOW))
ax.set_xticklabels(tokens_str[:N_SHOW], rotation=90, fontsize=7)
ax.set_yticklabels(tokens_str[:N_SHOW], fontsize=7)
plt.tight_layout()
plt.show()

## Entropy of attention distributions

Low entropy → the head attends sharply to a few positions (specialised).
High entropy → the head spreads attention broadly.

In [ ]:
def attention_entropy(attn_weights):
    """Mean entropy of attention distribution per head (averaged over query positions)."""
    w = attn_weights[0].detach().cpu().numpy()   # (heads, seq, seq)
    eps = 1e-9
    H = -(w * np.log(w + eps)).sum(axis=-1)      # (heads, seq)
    return H.mean(axis=-1)                       # (heads,)

print(f"{'Layer':>6}  {'Head':>5}  {'Mean entropy':>12}")
print("-" * 28)
for layer_idx, attn in enumerate(all_attn_weights):
    for head_idx, e in enumerate(attention_entropy(attn)):
        print(f"{layer_idx:>6}  {head_idx:>5}  {e:>12.3f}")